In [1]:
import pandas as pd
import json
import requests

In [26]:
def pullFromIPAWS(url):
    data=""
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"An error occurred: {err}")
    return data
def getData(data):
    rows = []
    for alert in data["IpawsArchivedAlerts"]:
        infos = alert.get("info") or []
        for info in infos:
            area_descs = [
                area.get("areaDesc")
                for area in info.get("areas", [])
                if area.get("areaDesc")
            ]
    
            rows.append({
                "identifier": alert.get("identifier"),
                "sent": alert.get("sent"),
                "status": alert.get("status"),
                "msgType": alert.get("msgType"),
    
                "senderName": info.get("senderName"),
                "event": info.get("event"),
                "responseType": info.get("responseType"),
                "urgency": info.get("urgency"),
                "severity": info.get("severity"),
                "certainty": info.get("certainty"),
    
                "headline": info.get("headline"),
                "description": info.get("description"),
                "instruction": info.get("instruction"),
    
                "areaDesc": "; ".join(area_descs),
    
            })
    
    df = pd.DataFrame(rows)
    return df
def cleanDf(df):
    df["date"] = pd.to_datetime(df["sent"]).dt.date
    df["time"] = pd.to_datetime(df["sent"]).dt.time
    
    df = df.dropna(subset=['description','urgency','severity','certainty'])
    df = df.drop_duplicates(subset=['senderName','date','description','status', 'event','urgency','severity','certainty'])
    df["description_clean"] = (
        df["description"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )
    
    df = df.drop_duplicates(subset=["description_clean"])
    return df

In [27]:
url1 = "https://www.fema.gov/api/open/v1/IpawsArchivedAlerts?$top=10000"
url2 = "https://www.fema.gov/api/open/v1/IpawsArchivedAlerts?$top=10000&$skip=10000"

In [28]:
data1=pullFromIPAWS(url1)
data2=pullFromIPAWS(url2)
df1=getData(data1)
df2=getData(data2)
df=pd.concat([df1, df2], ignore_index=True)
concat_df=cleanDf(df)

In [32]:
concat_df

,identifier,sent,status,msgType,senderName,event,responseType,urgency,severity,certainty,headline,description,instruction,areaDesc,date,time,description_clean
0,NWS-130404-684997,2013-12-28T20:36:38.000Z,Actual,Alert,NWS Wichita KS,Extreme Fire Danger,[Prepare],Expected,Moderate,Likely,Extreme Fire Danger issued December 28 at 2:36...,THE GFDI DOES NOT ACCOUNT FOR PRECIPITATION TH...,,Marion,2013-12-28,20:36:38,the gfdi does not account for precipitation th...
1,NWS-130404-687150-475299,2013-12-30T05:56:52.000Z,Actual,Cancel,NWS Taunton MA,Wind Advisory,[Execute],Expected,Moderate,Likely,Wind Advisory issued December 30 at 12:56AM ES...,THE NATIONAL WEATHER SERVICE IN TAUNTON HAS CA...,,Nantucket; Dukes; Barnstable,2013-12-30,05:56:52,the national weather service in taunton has ca...
2,NWS-130404-686163-474515,2013-12-29T15:00:42.000Z,Actual,Alert,NWS Shreveport LA,Flood Warning,[Avoid],Expected,Severe,Likely,Flood Warning issued December 29 at 9:00AM CST...,THE NATIONAL WEATHER SERVICE IN SHREVEPORT HAS...,,Bossier; Red River; Bienville; Webster,2013-12-29,15:00:42,the national weather service in shreveport has...
3,NWS-130404-687428-475456,2013-12-30T10:05:05.000Z,Actual,Update,NWS Chicago IL,Wind Chill Advisory,[Prepare],Expected,Moderate,Likely,Wind Chill Advisory issued December 30 at 4:04...,* MINIMUM WIND CHILL VALUES...15 TO 25 BELOW Z...,A WIND CHILL ADVISORY MEANS THAT THE WIND WILL...,Ogle; Kane; Kendall; La Salle; Livingston; McH...,2013-12-30,10:05:05,* minimum wind chill values...15 to 25 below z...
4,urn:oid:2.49.0.1.840.0.b65cf952732647052c047cb...,2022-12-22T04:28:00.000Z,Actual,Update,NWS Denver CO,Winter Weather Advisory,[Execute],Expected,Moderate,Likely,Winter Weather Advisory issued December 21 at ...,"* WHAT...Winds gusting as high as 50 mph, snow...",The combination of wind and cold temperatures ...,,2022-12-22,04:28:00,"* what...winds gusting as high as 50 mph, snow..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20005,NWS-IDP-PROD-4265555-3593476,2020-06-09T16:12:00.000Z,Actual,Alert,NWS Paducah KY,Severe Thunderstorm Warning,[Shelter],Immediate,Severe,Observed,Severe Thunderstorm Warning issued June 9 at 1...,The National Weather Service in Paducah has is...,For your protection move to an interior room o...,"Pope, IL; Johnson, IL; Massac, IL",2020-06-09,16:12:00,the national weather service in paducah has is...
20007,NWS-IDP-PROD-4265605,2020-06-09T16:38:00.000Z,Actual,Alert,NWS Tallahassee FL,Special Weather Statement,[Execute],Expected,Moderate,Observed,Special Weather Statement issued June 9 at 11:...,"At 1138 AM CDT, Doppler radar was tracking a s...",None,Coffee; Dale,2020-06-09,16:38:00,"at 1138 am cdt, doppler radar was tracking a s..."
20008,NWS-IDP-PROD-4265494-3593434,2020-06-09T15:48:00.000Z,Actual,Alert,NWS Sioux Falls SD,Wind Advisory,[Execute],Expected,Moderate,Likely,Wind Advisory issued June 9 at 10:48AM CDT unt...,* WHAT...Northwest winds 25 to 35 mph with gus...,"Use extra caution when driving, especially if ...",Minnehaha; Turner; Hutchinson; Lincoln; Kingsb...,2020-06-09,15:48:00,* what...northwest winds 25 to 35 mph with gus...
20009,NWS-IDP-PROD-4265616-3593530,2020-06-09T16:43:00.000Z,Actual,Update,NWS St Louis MO,Flood Warning,[Avoid],Immediate,Severe,Observed,Flood Warning issued June 9 at 11:43AM CDT by ...,...The flood warning is cancelled for the foll...,Rainfall heavier than forecast could cause riv...,"Pike, MO; Pike, IL",2020-06-09,16:43:00,...the flood warning is cancelled for the foll...


In [29]:
dupes = concat_df[concat_df.duplicated(subset=["description"], keep=False)]
dupes

,identifier,sent,status,msgType,senderName,event,responseType,urgency,severity,certainty,headline,description,instruction,areaDesc,date,time,description_clean


In [30]:
concat_df.to_csv("sample_IPAWS_data.csv",index=False)

In [33]:
classes = ["Minor", "Moderate", "Severe", "Extreme"]
n = 250  
balanced = (
    concat_df[concat_df["severity"].isin(classes)]
    .groupby("severity", group_keys=False)
    .sample(n=n, random_state=42)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
balanced
balanced['severity'].value_counts()

severity
Moderate    250
Minor       250
Severe      250
Extreme     250
Name: count, dtype: int64

In [34]:
balanced.to_csv("balanced_sample.csv", index=False)

In [41]:
natural_sample=concat_df[concat_df['severity']!="Unknown"]
natural_sample = natural_sample.sample(1000)
natural_sample['severity'].value_counts()

severity
Moderate    503
Severe      405
Minor        72
Extreme      20
Name: count, dtype: int64

In [42]:
natural_sample.to_csv("natural_sample.csv", index=False)